# Classical Image Processing with scikit-image and OpenCV

Before a neural network ever sees an image, that image is just an array of numbers — height ×
width × color channels. This notebook works directly on that array using classical (non-learned)
image operations: resizing, gray-scale conversion, contrast adjustment, thresholding, color
histograms, segmentation, blurring, and edge detection. Every one of these is a fixed recipe that
someone designed by hand. The point of the rest of Unit 2 is that a convolutional network *learns*
these recipes instead — so it pays to know what the hand-built versions do first.

## Learning objectives

- Load an image with OpenCV and explain why its channel order must be converted to RGB.
- Resize, crop, and convert images to gray-scale, tracking the array shape at each step.
- Apply a gamma transform to brighten or darken an image, and predict its effect on contrast.
- Compare global, Otsu, and adaptive thresholding on the same image.
- Read RGB and HSV histograms and say what each channel encodes.
- Segment an image by clustering its pixel colors with k-means and a Gaussian mixture.
- Remove noise with Gaussian and median blurs, and explain when each one wins.
- Detect edges with a Sobel filter, then design and apply a convolution kernel of your own.

## Background

An image in memory is a NumPy array. A color image has shape `(height, width, 3)`, where the last
axis holds the red, green, and blue intensity of each pixel; a gray-scale image drops that axis and
is simply `(height, width)`. Pixel intensities are stored either as 8-bit unsigned integers in
$[0, 255]$ (`uint8`) or as floats in $[0, 1]$ — many library functions assume one or the other, so
several cells below convert between them deliberately.

One convention is worth memorizing up front: **OpenCV loads images in BGR order, not RGB.** Every
`cv2.imread` below is therefore followed immediately by `cv2.cvtColor(..., cv2.COLOR_BGR2RGB)`.
Forget it and every image renders with its reds and blues swapped.

## This notebook covers

1. Loading and displaying an image
2. Resizing and cropping
3. Converting to gray-scale
4. Adjusting contrast with a gamma transform
5. Thresholding: global, Otsu, and adaptive
6. Color histograms in RGB and HSV
7. Segmentation by clustering pixel colors
8. Gaussian blur
9. Median blur
10. Edge detection with the Sobel filter
11. Designing and applying a custom kernel

**Prerequisites:** none — this is the first notebook of Unit 2.

**Dataset:** a single still image read from the instructor's local `Datasets/Images/` folder
(`Logos.jpg` by default; several alternatives are commented out in section 1). Any JPG or PNG on
disk works — point `data_folder` and `image_path` at a file of your own.

**References:** https://scikit-image.org/docs/stable/ and https://docs.opencv.org/4.x/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Loading and displaying an image

`cv2.imread` reads a file from disk into a NumPy array. Two things to notice:

- **The shape.** A color image comes back as `(height, width, 3)` — one entry per pixel row, per
  pixel column, and per color channel.
- **The channel order.** OpenCV returns **BGR**, a historical quirk of the library. Matplotlib
  expects RGB, so we convert immediately with `cv2.cvtColor`. Comment that line out once and you
  will never forget it again.

Several image paths are listed below with all but one commented out — change which line is active
to run the whole notebook against a different picture.

In [ ]:
#!pip install opencv-python
import cv2

data_folder = 'C:/Users/Graham West/Python Notebooks/Meharry Teaching/Datasets/'

#image_path = 'Images/LakeLouise.jpg'
image_path = 'Images/Logos.jpg'
#image_path = 'Images/Bolognese.png'
#image_path = 'Images/QuantumBookPage.jpg'
#image_path = 'Images/DiversePeople.jpg'
#image_path = 'Images/P_01346_R_001.jpg'

image_path = data_folder + image_path

image = cv2.imread(image_path)
# OpenCV reads images as BGR; matplotlib expects RGB, so swap the channel order.
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
print(image.shape)

plt.figure(figsize=(10,10))
plt.imshow(image)
plt.show()

## 2. Resizing and cropping

Two different ways to make an image smaller, and they are not interchangeable.

**Resizing** rescales the whole picture onto a new pixel grid: the entire field of view survives,
but detail is thrown away. To avoid distorting the image we preserve its aspect ratio — fix a
target width $w'$ and derive the height from the original proportions,

$$ h' = w' \cdot \frac{h}{w} $$

**Cropping** takes a rectangular sub-array and discards everything outside it: full detail, smaller
field of view. Because the image is just a NumPy array, a crop is plain slicing.

The distinction matters later in the unit. CNNs need a fixed input size, so resize-versus-crop is a
real design decision when preparing a dataset.

### 2.1 Resize while preserving the aspect ratio

In [ ]:
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Resize while preserving aspect ratio
target_width = 500
h, w = image.shape[:2]
aspect_ratio = h / w
target_height = int(target_width * aspect_ratio)

image = cv2.resize(image, (target_width, target_height))
print(image.shape)

plt.figure(figsize=(10,10))
plt.imshow(image)
plt.show()

### 2.2 Crop a patch

Slicing `image[x:x+delta, y:y+delta, :]` keeps every pixel it selects at full resolution — no
interpolation, no detail lost, just a smaller window onto the scene.

The `delta = 224` is not arbitrary: 224×224 is the native input size of many pretrained ImageNet
models, including the MobileNetV2 used later in this unit.

In [ ]:
x = 40
y = 30
delta = 224

plt.figure(figsize=(10,10))
plt.imshow(image[x:x+delta,y:y+delta,:])
plt.show()

## 3. Converting to gray-scale

Dropping color collapses the three channels into a single intensity per pixel, so the array goes
from `(h, w, 3)` to `(h, w)`. `skimage.color.rgb2gray` does not use a plain average — it uses a
luminance-weighted sum,

$$ Y = 0.2125\,R + 0.7154\,G + 0.0721\,B $$

because human vision is far more sensitive to green than to blue. An unweighted mean would make
green objects look implausibly dark.

`rgb2gray` returns floats in $[0, 1]$, but the OpenCV thresholding functions in section 5 expect
`uint8`, so we rescale to $[0, 255]$ and change dtype here.

In [ ]:
from skimage import color

image_gray = color.rgb2gray(image)

# rgb2gray returns floats in [0, 1]; cv2's thresholding functions want uint8 in [0, 255].
if image_gray.dtype != np.uint8:
    image_gray = (image_gray * 255).astype(np.uint8)

plt.figure(figsize=(10,10))
plt.imshow(image_gray, cmap='gray')
plt.axis('off')
plt.show()

## 4. Adjusting contrast with a gamma transform

A **gamma transform** raises every normalized pixel intensity to a power:

$$ I' = I^{\gamma}, \qquad I \in [0, 1] $$

Because the input is confined to $[0,1]$, the exponent bends the intensity curve rather than merely
scaling it:

- $\gamma > 1$ **darkens** the image, pushing mid-tones down and expanding contrast among the
  bright pixels.
- $\gamma < 1$ **brightens** it, expanding contrast among the dark pixels instead.
- $\gamma = 1$ leaves the image untouched.

Normalizing first is essential — the transform only behaves this way on $[0,1]$. Change `gamma`
below and watch which parts of the image gain detail and which get crushed.

### 4.1 Gamma on the gray-scale image

In [ ]:
# normalize
#image_gray_gamme = image_gray / 255
image_gray_gamme = image_gray / image_gray.max()

gamma = 3

image_gray_gamme = image_gray_gamme ** gamma

plt.figure(figsize=(10,10))
plt.imshow(image_gray_gamme, cmap='gray')
plt.show()

### 4.2 The same transform on the color image

Applying the identical exponent to all three channels changes brightness and contrast while leaving
the *relative* proportions of R, G, and B at each pixel roughly intact, so colors stay recognizable
rather than shifting hue.

Here the normalize → exponentiate → rescale round trip happens in a single line, and the result is
cast back to `uint8` for display.

In [ ]:
image_gamma = np.floor(255 * ((image / 255.0) ** gamma)).astype(np.uint8)

plt.figure(figsize=(10,10))
plt.imshow(image_gamma)


## 5. Thresholding: global, Otsu, and adaptive

Thresholding turns a gray-scale image into a binary one by asking a yes/no question of every pixel.
The three methods differ only in where the threshold $T$ comes from.

**Global.** One fixed $T$ for the entire image, chosen by hand:

$$ I'(x,y) = \begin{cases} 255 & \text{if } I(x,y) > T \\ 0 & \text{otherwise} \end{cases} $$

**Otsu.** Picks $T$ automatically: it tries every candidate value and keeps the one that minimizes
the intensity variance *within* the two resulting groups — equivalently, that maximizes the variance
*between* them. It works well when the histogram is bimodal (a distinct foreground and background)
and poorly when it is not.

**Adaptive.** Computes a *different* threshold for every pixel from its own neighborhood — the mean
or a Gaussian-weighted mean of the surrounding `kernel_size` × `kernel_size` window, minus a small
constant. This is the method that survives uneven lighting, where no single global $T$ can be right
everywhere at once.

Compare the four results below, paying particular attention to any region where the illumination
changes across the frame.

In [ ]:
thresh      = 30
kernel_size = 31

# Thresholding methods
ret1, th1 = cv2.threshold(image_gray, thresh, 255, cv2.THRESH_BINARY)
ret2, th2 = cv2.threshold(image_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
th3 = cv2.adaptiveThreshold(image_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                            cv2.THRESH_BINARY, kernel_size, 2)
th4 = cv2.adaptiveThreshold(image_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                            cv2.THRESH_BINARY, kernel_size, 2)

# Titles and image list
titles = [
    'Original Image',
    f'Global Thresh (thresh={thresh})',
    'Otsu Thresh',
    'Adaptive Mean Thresh',
    'Adaptive Gaussian Thresh'
]
images = [image_gray, th1, th2, th3, th4]

# Create 2x3 subplot grid
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i in range(5):
    axes[i].imshow(images[i], cmap='gray')
    axes[i].set_title(titles[i])
    axes[i].axis('off')

# Hide the last (unused) subplot
axes[5].axis('off')

plt.tight_layout()
plt.show()


## 6. Color histograms in RGB and HSV

A histogram counts how many pixels fall into each intensity bin, discarding all spatial information
— it says *what* colors are present, never *where*. Two color spaces give very different views of
the same picture:

- **RGB** stores additive primaries. The three channels are strongly correlated (brightening an
  image shifts all three at once), which makes a request like "find everything red" awkward to
  express.
- **HSV** separates **hue** (which color, as an angle around the color wheel), **saturation** (how
  vivid versus washed out), and **value** (how bright). Because hue is largely independent of
  lighting, HSV is usually the better space for color-based selection.

All six histograms below share a common y-limit so their heights can be compared directly.

### 6.1 Channel histograms

In [ ]:
image_float = image.astype(np.float32) / 255.0

bins = 256 // 4

# ---------------------------
# Compute Histograms for RGB
# ---------------------------
r_hist, r_bins = np.histogram(image_float[:, :, 0], bins=bins, range=(0, 1))
g_hist, g_bins = np.histogram(image_float[:, :, 1], bins=bins, range=(0, 1))
b_hist, b_bins = np.histogram(image_float[:, :, 2], bins=bins, range=(0, 1))

# ---------------------------
# Convert Image to HSV and Compute Histograms
# ---------------------------
image_float_hsv = color.rgb2hsv(image_float)

h_hist, h_bins = np.histogram(image_float_hsv[:, :, 0], bins=bins, range=(0, 1))
s_hist, s_bins = np.histogram(image_float_hsv[:, :, 1], bins=bins, range=(0, 1))
v_hist, v_bins = np.histogram(image_float_hsv[:, :, 2], bins=bins, range=(0, 1))

# Compute the maximum frequency across all histograms for a uniform y-limit
max_val = max(np.max(r_hist), np.max(g_hist), np.max(b_hist),
              np.max(h_hist), np.max(s_hist), np.max(v_hist))

# ---------------------------
# Plotting: 3 rows x 2 columns (each channel on a separate subplot)
# ---------------------------
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 12))

# Row 1: Red (RGB) and Hue (HSV)
axes[0, 0].plot(r_bins[:-1], r_hist, color='r')
axes[0, 0].set_title('Red Histogram (RGB)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_ylim(0, max_val)

axes[0, 1].plot(h_bins[:-1], h_hist, color='w')
axes[0, 1].set_title('Hue Histogram (HSV)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_ylim(0, max_val)

# Row 2: Green (RGB) and Saturation (HSV)
axes[1, 0].plot(g_bins[:-1], g_hist, color='g')
axes[1, 0].set_title('Green Histogram (RGB)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_ylim(0, max_val)

axes[1, 1].plot(s_bins[:-1], s_hist, color='w')
axes[1, 1].set_title('Saturation Histogram (HSV)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_ylim(0, max_val)

# Row 3: Blue (RGB) and Value (HSV)
axes[2, 0].plot(b_bins[:-1], b_hist, color='b')
axes[2, 0].set_title('Blue Histogram (RGB)')
axes[2, 0].set_ylabel('Frequency')
axes[2, 0].set_ylim(0, max_val)

axes[2, 1].plot(v_bins[:-1], v_hist, color='w')
axes[2, 1].set_title('Value Histogram (HSV)')
axes[2, 1].set_ylabel('Frequency')
axes[2, 1].set_ylim(0, max_val)

plt.tight_layout()
plt.show()

### 6.2 The channels as images

The same six channels, displayed as gray-scale images rather than histograms — this restores the
spatial information the histograms threw away.

The HSV panels are the interesting ones. Watch how the saturation channel isolates vivid regions
regardless of their color, and how the value channel closely resembles the plain gray-scale
conversion from section 3.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(10, 15))
axes = axes.ravel()

# Row 1: original image (span both columns)
axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")
axes[1].axis("off")  # empty to span layout

# Rows 2–4: RGB (left) and HSV (right)
channels_rgb = ['Red', 'Green', 'Blue']
channels_hsv = ['Hue', 'Saturation', 'Value']

for i in range(3):
    # RGB channels
    axes[2 * (i + 1)].imshow(image[:, :, i], cmap='gray')
    axes[2 * (i + 1)].set_title(f"RGB - {channels_rgb[i]}")
    axes[2 * (i + 1)].axis("off")
    
    # HSV channels
    axes[2 * (i + 1) + 1].imshow(image_float_hsv[:, :, i], cmap='gray')
    axes[2 * (i + 1) + 1].set_title(f"HSV - {channels_hsv[i]}")
    axes[2 * (i + 1) + 1].axis("off")

plt.tight_layout()
plt.show()

## 7. Segmentation by clustering pixel colors

**Segmentation** partitions an image into regions. One simple way to do it: forget where the pixels
are entirely, treat each one as a point $(r, g, b)$ in three-dimensional color space, and cluster
those points.

**k-means** looks for $k$ cluster centers $\mu_1, \dots, \mu_k$ that minimize the total squared
distance from every pixel to its assigned center:

$$ \sum_{i=1}^{n} \left\lVert x_i - \mu_{c(i)} \right\rVert^2 $$

Repainting each pixel with its center's color yields a posterized image using only $k$ distinct
colors.

**Gaussian mixtures** fit $k$ Gaussian blobs instead of $k$ points, so a cluster can be wide in one
color direction and narrow in another. The result is softer boundaries between segments.

Because position is ignored, a "segment" here is a set of similarly-colored pixels that may be
scattered all over the frame — this is color quantization more than object detection. Change
`n_clusters` to trade fidelity against simplification.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

# Number of clusters
n_clusters = 5

# Prepare clustering methods
methods = []

# Get shape
h, w, c = image.shape

# Reshape to (num_pixels, 3)
pixels = image.reshape(-1, 3)
print(pixels.shape)

# KMeans
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
k_labels = kmeans.fit_predict(pixels)
k_centers = kmeans.cluster_centers_
methods.append(('KMeans', k_labels, k_centers))

# GMM
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
g_labels = gmm.fit_predict(pixels)
g_centers = gmm.means_
methods.append(('GMM', g_labels, g_centers))

# Plot each clustering method in its own figure
for name, labels, centers in methods:
    segmented = centers[labels].astype(np.uint8).reshape((h, w, 3))
    labels_img = labels.reshape((h, w))
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(name, fontsize=16)

    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(labels_img, cmap='gray')
    axes[1].set_title('Cluster Labels')
    axes[1].axis('off')

    axes[2].imshow(segmented)
    axes[2].set_title('Centroid Colors')
    axes[2].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## 8. Gaussian blur

A **Gaussian blur** replaces each pixel with a weighted average of its neighbors, the weights
falling off with distance according to

$$ G(x, y) = \frac{1}{2\pi\sigma^2}\, e^{-\frac{x^2 + y^2}{2\sigma^2}} $$

This is a **convolution** — the operation at the heart of every CNN in the rest of this unit — with
a fixed, hand-designed kernel. A convolutional network runs exactly this machinery, except that the
kernel weights are learned from data instead of written down in advance.

To make the effect visible we first corrupt the image with additive Gaussian noise, then blur it at
increasing kernel sizes. Averaging cancels zero-mean noise, so the speckle fades — but genuine edges
fade with it, because a linear filter cannot tell signal from noise. That trade-off between
denoising and blurring is the whole story of linear smoothing.

In [ ]:
noise = np.random.normal(loc=0, scale=100, size=image.shape)
noisy_image = image.astype(float) + noise.astype(np.int16)
noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)

# Define kernel sizes for Gaussian blurring (must be odd)
kernel_sizes = [7, 11, 15, 31, 51]

# Set up figure (2 rows, 3 columns)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Original image
axes[0].imshow(noisy_image)
axes[0].set_title("Original (Noisy)")
axes[0].axis("off")

# Apply Gaussian blur for each kernel size and plot
for i, k in enumerate(kernel_sizes, start=1):
    blurred = cv2.GaussianBlur(noisy_image, (k, k), 0)
    axes[i].imshow(blurred)
    axes[i].set_title(f"Gaussian Blur ({k}×{k})")
    axes[i].axis("off")

# Hide any unused subplot (if fewer than 6 images)
for j in range(len(kernel_sizes) + 1, 6):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 9. Median blur

A **median blur** replaces each pixel with the *median* of its neighborhood instead of the mean.
That is not a convolution — the median is nonlinear, and no set of fixed kernel weights can produce
it.

The nonlinearity buys something real. A single wildly wrong pixel drags a mean toward itself, but
barely moves a median, so median filtering removes salt-and-pepper noise almost perfectly while
keeping edges crisp.

Compare these results against section 8's at the same kernel sizes: the Gaussian softens edges as it
denoises, and the median largely does not.

In [ ]:
noise = np.random.normal(loc=0, scale=100, size=image.shape)
noisy_image = image.astype(float) + noise.astype(np.int16)
noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)

# Define kernel sizes for median blurring (must be odd)
kernel_sizes = [7, 11, 15, 31, 51]

# Set up figure (2 rows, 3 columns)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Original image
axes[0].imshow(noisy_image)
axes[0].set_title("Original (Noisy)")
axes[0].axis("off")

# Apply median blur for each kernel size and plot
for i, k in enumerate(kernel_sizes, start=1):
    blurred = cv2.medianBlur(noisy_image, k)
    axes[i].imshow(blurred)
    axes[i].set_title(f"Median Blur ({k}×{k})")
    axes[i].axis("off")

# Hide any unused subplot (if fewer than 6 images)
for j in range(len(kernel_sizes) + 1, 6):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 10. Edge detection with the Sobel filter

An **edge** is a place where intensity changes rapidly, so edge detection is fundamentally a
question about derivatives. The Sobel filter estimates the image gradient by convolving with two
small kernels — one sensitive to horizontal change, one to vertical:

$$ K_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}, \qquad
   K_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} $$

`filters.sobel` returns the gradient **magnitude**, which combines the two into a single
orientation-independent measure of edge strength:

$$ \lVert \nabla I \rVert = \sqrt{(I * K_x)^2 + (I * K_y)^2} $$

Bright pixels in the result are strong edges. Note that this is one more hand-designed kernel — and
that the early layers of a trained CNN reliably learn filters that look very much like these,
without anyone having told them to.

In [ ]:
from skimage import filters

edges = filters.sobel(image_gray)

plt.figure(figsize=(10,10))
plt.imshow(edges, cmap='gray')
plt.axis('off')
plt.show()

## 11. Designing and applying a custom kernel

Every filter in sections 8 and 10 was a convolution with a kernel someone else chose. Here we choose
one ourselves and see what it responds to. Two-dimensional convolution is

$$ (I * K)[i, j] = \sum_{m}\sum_{n} I[i+m,\; j+n] \; K[m, n] $$

— slide $K$ across the image and, at each position, record the weighted sum of the pixels underneath
it. Because that sum is largest where the local neighborhood *resembles* the kernel, a kernel acts
as a template detector for its own pattern.

The active kernel below is a vertical sinusoid, so it fires most strongly on stripes at that
frequency and orientation. Several alternatives are commented out — uniform averaging (a blur), a
diagonal streak (motion blur), and random kernels. Try them, and predict before each run which parts
of the image will light up.

Two arguments worth noting: `mode='same'` keeps the output the same size as the input, and
`boundary='fill'` with `fillvalue=0` pads the borders with zeros — the very same **zero padding** a
Keras `Conv2D` layer applies when you write `padding='same'`.

In [ ]:
from scipy.signal import convolve2d

# Define the kernel size k (for example, 5x5)
k = 11

# Create a custom kernel of size k x k.
# For example, here we create a uniform averaging kernel:
#kernel = np.ones((k, k))
#kernel = kernel / kernel.sum()

#kernel = np.diag( [1]*k )
#kernel = kernel / kernel.sum()
#kernel = kernel[::-1,:]

#kernel = np.random.rand(k,k)
#kernel = kernel / kernel.sum()

#kernel = np.random.randn(k,k)
#kernel[0,0] = 10
#kernel[8,3] = -10
#kernel[5,10] = 10

freq = 2
kernel = -np.repeat( np.sin( 2 * np.pi * freq * np.arange(k) / k ), k ).reshape(k,k).T

plt.figure(figsize=(5, 5))
im = plt.imshow(kernel, interpolation='none', cmap='viridis')
plt.title("Custom Kernel")
plt.colorbar(im)
plt.axis('off')
plt.show()

# Apply the kernel to image_gray using zero padding.
# Using scipy.signal.convolve2d: mode='same' returns an output the same size as input,
# and boundary='fill' with fillvalue=0 applies zero padding.
convolved_image = convolve2d(image_gray, kernel, mode='same', boundary='fill', fillvalue=0)

# Plot the original and convolved images
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
im2 = plt.imshow(image_gray, cmap='gray')
plt.colorbar(im2)
plt.title('Original Grayscale Image')

plt.subplot(1, 2, 2)
im3 = plt.imshow(convolved_image, cmap='gray')
plt.colorbar(im3)
plt.title('Image convolved with kernel')

plt.tight_layout()
plt.show()

## 12. Review

| Operation | Function used | What it does | Linear? |
|---|---|---|---|
| Load / color convert | `cv2.imread`, `cv2.cvtColor` | File → array; BGR → RGB | — |
| Resize | `cv2.resize` | Rescales onto a new pixel grid | — |
| Crop | NumPy slicing | Keeps a sub-rectangle at full detail | — |
| Gray-scale | `color.rgb2gray` | Luminance-weighted channel collapse | yes |
| Gamma | `** gamma` on $[0,1]$ | Bends the intensity curve | no |
| Threshold | `cv2.threshold`, `cv2.adaptiveThreshold` | Gray → binary, globally or locally | no |
| Histogram | `np.histogram` | Color content, spatial info discarded | — |
| Segmentation | `KMeans`, `GaussianMixture` | Groups pixels by color | no |
| Gaussian blur | `cv2.GaussianBlur` | Distance-weighted neighborhood mean | yes |
| Median blur | `cv2.medianBlur` | Neighborhood median | no |
| Sobel edges | `filters.sobel` | Gradient magnitude | yes |
| Custom kernel | `convolve2d` | Any convolution you design | yes |

**Takeaways**

- **An image is just an array.** Every operation here is arithmetic on `(h, w, 3)` or `(h, w)`
  numbers — cropping is slicing, gamma is exponentiation, blurring is a weighted sum.
- **Watch the dtype and the range.** `uint8` in $[0,255]$ and float in $[0,1]$ are both common, and
  library functions disagree about which they want. A large share of confusing image bugs are
  really conversion bugs.
- **Convolution is the thread running through this unit.** Gaussian blur, Sobel, and the custom
  kernel are all the same operation with different weights. What changes in the next notebook is
  *who picks the weights*: a CNN learns them from data rather than having them specified by hand,
  and the filters it learns in its early layers look strikingly like the edge detectors here.
- **Linear versus nonlinear matters.** The median blur beats the Gaussian blur on outlier noise
  precisely *because* it is not a convolution — no fixed set of kernel weights can ignore an
  extreme value the way a median does.

**Next:** `U2-2_CNN-1_DenseFails.ipynb` shows what goes wrong when a fully-connected network is
pointed at images, which is the argument for convolutional layers.